In [ ]:
%pip install openai
%pip install pypdf

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pypdf import PdfReader

def cargar_pdf(path):
    """Lee el texto de un PDF página por página"""
    reader = PdfReader(path)
    texto_paginas = []
    for i, page in enumerate(reader.pages):
        texto = page.extract_text()
        if texto:
            texto_paginas.append({"page": i+1, "text": texto})
    return texto_paginas

pdf_path = "Algebra_Superior_Rincon_Bravo.pdf"
paginas_pdf = cargar_pdf(pdf_path)

print(f"Se cargaron {len(paginas_pdf)} páginas")
print("Primeros caracteres de la página 1:\n")
print(paginas_pdf[0]["text"][:500])


Se cargaron 662 páginas
Primeros caracteres de la página 1:

Alejandro Bravo Mojica
Hugo Rincón Mejía
César Rincón Orta
ÁLGEBRA SUPERIOR
FACULTAD DECIENCIAS, UNAM
2013


In [ ]:
%pip install faiss-cpu

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install python-dotenv


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv

# Carga el archivo .env
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("MODEL_NAME", "gpt-4o-mini")
top_k = int(os.getenv("TOP_K", 5))

print("API Key loaded:", api_key is not None)
print("Model:", model_name)


API Key loaded: True
Model: gpt-4o-mini


In [ ]:
from openai import OpenAI
import faiss
import numpy as np

client = OpenAI(api_key=api_key)

def obtener_embeddings(textos):
    """Convierte una lista de strings en embeddings"""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=textos
    )
    return [e.embedding for e in response.data]

# Dividir en fragmentos (para más precisión en búsqueda)
documentos = []
for pagina in paginas_pdf:
    partes = [pagina["text"][i:i+800] for i in range(0, len(pagina["text"]), 800)]
    for parte in partes:
        documentos.append(parte)

# Generar embeddings
embeddings = obtener_embeddings(documentos)

# Crear índice FAISS
dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

print("Índice FAISS creado con", index.ntotal, "fragmentos")


Índice FAISS creado con 1302 fragmentos


In [ ]:
import os
import numpy as np
import faiss
from openai import OpenAI

# Inicia OpenAI client
api_key = os.getenv("OPENAI_API_KEY")  # lee desde .env o variable de entorno
client = OpenAI(api_key=api_key)

# Embeddings

def get_embeddings(texts):
    """
    Convert a list of strings into embeddings using OpenAI API.
    """
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [e.embedding for e in response.data]

# Dividir en fragmentos

def split_into_chunks(pages, chunk_size=800):
    """
    Split each page text into chunks.
    """
    chunks = []
    for page in pages:
        for i in range(0, len(page["text"]), chunk_size):
            chunks.append(page["text"][i:i + chunk_size])
    return chunks

def build_faiss_index(embeddings):
    """
    Build a FAISS index from embeddings.
    """
    dimension = len(embeddings[0])
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings).astype("float32"))
    print(f"FAISS index created with {index.ntotal} chunks")
    return index

# Busca dentro del índice para obtener contexto
def search_faiss(query, index, documents, top_k=5):
    """
    Search the FAISS index and return top_k relevant chunks.
    """
    query_emb = np.array(get_embeddings([query])).astype("float32")
    distances, indices = index.search(query_emb, top_k)
    return [documents[i] for i in indices[0]]

# prompt para respuesta

def build_prompt(base_prompt, context_chunks, question):
    """
    Combine base prompt with context chunks and student question.
    """
    context_text = "\n\n".join(context_chunks)
    return f"""{base_prompt}

Relevant context from PDF:
{context_text}

Student question:
{question}
"""

def answer_question(question, index, documents, base_prompt, chat_history=None, top_k=5):
    """
    Answer the student's question using FAISS context and chat history.
    """
    context_chunks = search_faiss(question, index, documents, top_k=top_k)
    prompt = build_prompt(base_prompt, context_chunks, question)

    messages = []
    if chat_history:
        messages.extend(chat_history)  # mantiene el hilo
    messages.append({"role": "system", "content": prompt})
    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.2
    )

    answer = response.choices[0].message.content
    if chat_history is not None:
        chat_history.append({"role": "user", "content": question})
        chat_history.append({"role": "assistant", "content": answer})

    return answer

# paginas del PDF (ejemplo)
paginas_pdf = [
    {"text": "Contenido de la página 1 de álgebra lineal..."},
    {"text": "Contenido de la página 2 de álgebra lineal..."}
]

documents = split_into_chunks(paginas_pdf)
embeddings = get_embeddings(documents)
index = build_faiss_index(embeddings)

# Base prompt
base_prompt = """
Eres un asistente virtual educativo diseñado para apoyar a estudiantes de la licenciatura en Matemáticas a distancia. Tu objetivo principal es ayudar a los estudiantes a comprender conceptos, repasar contenidos y desarrollar sus propias soluciones, sin darles respuestas directas a sus tareas o evaluaciones.

Debes ser amable, paciente y motivador, fomentando el pensamiento crítico y la autonomía. Si un estudiante tiene dudas, primero ayúdalo a razonar con explicaciones paso a paso, ejemplos similares o preguntas guía.

Si detectas que el estudiante necesita más apoyo, redirígelo a recursos adicionales disponibles en la plataforma Moodle, como:

Notas escritas y materiales de clase subidos por los profesores.
Videos explicativos recomendados en cada curso.
Ejercicios de repaso o foros de discusión.
Temas en los que puedes apoyar:

Cálculo diferencial e integral
Álgebra lineal
Geometría
Otros cursos disponibles en el plan de estudios
Normas clave:

No entregues la solución final a ejercicios, tareas o exámenes.
No realices pasos completos que impliquen resolver un problema por el estudiante.
Siempre explica de manera clara, usando ejemplos generales cuando sea necesario.
Incentiva al estudiante a reflexionar y verificar sus resultados.
Promueve el uso de los materiales oficiales del curso.
Usa un tono cordial y cercano, adaptándote al nivel de cada estudiante. Tu meta final es que el estudiante aprenda a aprender y adquiera confianza en sus capacidades.
"""



FAISS index created with 2 chunks


In [ ]:
chat_history = [] # lista de mensajes previos

# Ejemplo de preguntas consecutivas
question1 = "¿Qué significa que un conjunto de vectores sea linealmente independiente?"
answer1 = answer_question(question1, index, documents, base_prompt, chat_history)
print(answer1)

question2 = "¿Cómo puedo determinar si una matriz es invertible?"
answer2 = answer_question(question2, index, documents, base_prompt, chat_history)
print(answer2)


In [ ]:
question3 = "¿Qué es un espacio vectorial?"
answer3 = answer_question(question3, index, documents, base_prompt, chat_history)
print(answer3)

Un **espacio vectorial** es una estructura matemática que consiste en un conjunto de vectores, los cuales pueden ser sumados entre sí y multiplicados por escalares (números reales o complejos, dependiendo del contexto). Para que un conjunto y las operaciones definidas en él formen un espacio vectorial, deben cumplir ciertas propiedades o axiomas.

### Definición Formal:
Un conjunto \( V \) se llama espacio vectorial sobre un campo \( F \) (como los números reales \( \mathbb{R} \) o los números complejos \( \mathbb{C} \)) si se cumplen las siguientes condiciones:

1. **Cerradura bajo la suma**: Si \( \mathbf{u} \) y \( \mathbf{v} \) son vectores en \( V \), entonces \( \mathbf{u} + \mathbf{v} \) también está en \( V \).
   
2. **Cerradura bajo la multiplicación por escalares**: Si \( \mathbf{u} \) es un vector en \( V \) y \( c \) es un escalar en \( F \), entonces \( c\mathbf{u} \) también está en \( V \).

3. **Asociatividad de la suma**: Para todos los vectores \( \mathbf{u}, \mathbf

In [ ]:
import json
import os

# Configuración del cache
CACHE_FILE = "chat_cache.json"

# Cargar cache existente o crear uno vacío
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        chat_cache = json.load(f)
else:
    chat_cache = {}

# Función para obtener respuesta con cache
def get_answer_with_cache(question, index, documents, base_prompt, chat_history):
    # Revisar si la pregunta ya está en cache
    if question in chat_cache:
        print("Respuesta obtenida del cache")
        return chat_cache[question]

    # Si no está en cache, obtener la respuesta
    answer = answer_question(question, index, documents, base_prompt, chat_history)

    # Guardar en cache
    chat_cache[question] = answer
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(chat_cache, f, ensure_ascii=False, indent=2)

    return answer

# Ejemplo de uso
chat_history = []

question4 = "¿Qué significa que un conjunto de vectores sea linealmente independiente?"
answer4 = get_answer_with_cache(question4, index, documents, base_prompt, chat_history)
print(answer4)



Respuesta obtenida del cache
¡Excelente pregunta! La independencia lineal es un concepto fundamental en álgebra lineal. Un conjunto de vectores se dice que es linealmente independiente si ninguno de esos vectores puede ser expresado como una combinación lineal de los otros vectores del conjunto.

Para entenderlo mejor, consideremos lo siguiente:

1. **Combinación lineal**: Un vector \( \mathbf{v} \) se puede expresar como una combinación lineal de otros vectores \( \mathbf{u_1}, \mathbf{u_2}, \ldots, \mathbf{u_n} \) si existen escalares \( c_1, c_2, \ldots, c_n \) tales que:
   \[
   \mathbf{v} = c_1 \mathbf{u_1} + c_2 \mathbf{u_2} + \ldots + c_n \mathbf{u_n}
   \]

2. **Independencia lineal**: Un conjunto de vectores \( \{\mathbf{u_1}, \mathbf{u_2}, \ldots, \mathbf{u_n}\} \) es linealmente independiente si la única solución a la ecuación:
   \[
   c_1 \mathbf{u_1} + c_2 \mathbf{u_2} + \ldots + c_n \mathbf{u_n} = \mathbf{0}
   \]
   es \( c_1 = c_2 = \ldots = c_n = 0 \). Esto significa

In [ ]:
question5 = "¿Qué significa que un conjunto de vectores sea linealmente independiente?"
answer5 = get_answer_with_cache(question5, index, documents, base_prompt, chat_history)
print(answer5)


Respuesta obtenida del cache
¡Excelente pregunta! La independencia lineal es un concepto fundamental en álgebra lineal. Un conjunto de vectores se dice que es linealmente independiente si ninguno de esos vectores puede ser expresado como una combinación lineal de los otros vectores del conjunto.

Para entenderlo mejor, consideremos lo siguiente:

1. **Combinación lineal**: Un vector \( \mathbf{v} \) se puede expresar como una combinación lineal de otros vectores \( \mathbf{u_1}, \mathbf{u_2}, \ldots, \mathbf{u_n} \) si existen escalares \( c_1, c_2, \ldots, c_n \) tales que:
   \[
   \mathbf{v} = c_1 \mathbf{u_1} + c_2 \mathbf{u_2} + \ldots + c_n \mathbf{u_n}
   \]

2. **Independencia lineal**: Un conjunto de vectores \( \{\mathbf{u_1}, \mathbf{u_2}, \ldots, \mathbf{u_n}\} \) es linealmente independiente si la única solución a la ecuación:
   \[
   c_1 \mathbf{u_1} + c_2 \mathbf{u_2} + \ldots + c_n \mathbf{u_n} = \mathbf{0}
   \]
   es \( c_1 = c_2 = \ldots = c_n = 0 \). Esto significa

In [ ]:
question6 = "¿Que significa que un conjunto de vectores sea linealmente independiente?"
answer6 = get_answer_with_cache(question6, index, documents, base_prompt, chat_history)
print(answer6)

Un conjunto de vectores es linealmente independiente si ninguno de los vectores en el conjunto puede ser expresado como una combinación lineal de los otros vectores. Esto significa que no puedes tomar algunos de los vectores, multiplicarlos por escalares y sumarlos para obtener otro vector del mismo conjunto, excepto en el caso trivial donde todos los escalares son cero.

### Para entenderlo mejor, considera lo siguiente:

1. **Combinación lineal**: Un vector \( \mathbf{v} \) se puede escribir como una combinación lineal de otros vectores \( \mathbf{u_1}, \mathbf{u_2}, \ldots, \mathbf{u_n} \) si existen números (escalares) \( c_1, c_2, \ldots, c_n \) tales que:
   \[
   \mathbf{v} = c_1 \mathbf{u_1} + c_2 \mathbf{u_2} + \ldots + c_n \mathbf{u_n}
   \]

2. **Independencia lineal**: Un conjunto de vectores \( \{\mathbf{u_1}, \mathbf{u_2}, \ldots, \mathbf{u_n}\} \) es linealmente independiente si la única solución a la ecuación:
   \[
   c_1 \mathbf{u_1} + c_2 \mathbf{u_2} + \ldots + c_n 

In [ ]:
question7 = "¿De que trata el primer cápitulo del libro Algebra Superior de Rincon y Bravo?"
answer7 = get_answer_with_cache(question7, index, documents, base_prompt, chat_history)
print(answer7)

El primer capítulo del libro "Álgebra Superior" de Rincón y Bravo generalmente se enfoca en los conceptos fundamentales del álgebra lineal. Aunque no puedo proporcionarte un resumen exacto del contenido, puedo guiarte sobre los temas que suelen tratarse en este tipo de libros.

1. **Vectores y Espacios Vectoriales**: Se introduce el concepto de vectores, sus propiedades y operaciones básicas. También se define qué es un espacio vectorial y se discuten sus axiomas.

2. **Subespacios**: Se exploran los subespacios de un espacio vectorial, incluyendo criterios para determinar si un conjunto de vectores forma un subespacio.

3. **Combinaciones Lineales**: Se explica cómo se pueden combinar vectores y se introduce la noción de dependencia e independencia lineal.

4. **Bases y Dimensiones**: Se discute el concepto de base de un espacio vectorial y cómo se relaciona con la dimensión del espacio.

5. **Transformaciones Lineales**: Se puede incluir una introducción a las transformaciones lineal

In [ ]:
question8 = "¿De que trata el primer cápitulo del libro Algebra Superior de Rincon y Bravo?"
answer8 = get_answer_with_cache(question8, index, documents, base_prompt, chat_history)
print(answer8)

Respuesta obtenida del cache
El primer capítulo del libro "Álgebra Superior" de Rincón y Bravo generalmente se enfoca en los conceptos fundamentales del álgebra lineal. Aunque no puedo proporcionarte un resumen exacto del contenido, puedo guiarte sobre los temas que suelen tratarse en este tipo de libros.

1. **Vectores y Espacios Vectoriales**: Se introduce el concepto de vectores, sus propiedades y operaciones básicas. También se define qué es un espacio vectorial y se discuten sus axiomas.

2. **Subespacios**: Se exploran los subespacios de un espacio vectorial, incluyendo criterios para determinar si un conjunto de vectores forma un subespacio.

3. **Combinaciones Lineales**: Se explica cómo se pueden combinar vectores y se introduce la noción de dependencia e independencia lineal.

4. **Bases y Dimensiones**: Se discute el concepto de base de un espacio vectorial y cómo se relaciona con la dimensión del espacio.

5. **Transformaciones Lineales**: Se puede incluir una introducción 

In [ ]:
question9 = "De que trata el primer capitulo del libro Algebra Superior de Rincon y Bravo?"
answer9 = get_answer_with_cache(question9, index, documents, base_prompt, chat_history)
print(answer9)

El primer capítulo del libro "Álgebra Superior" de Rincón y Bravo generalmente se centra en los fundamentos del álgebra lineal. Aunque no puedo darte un resumen exacto, puedo mencionarte algunos de los temas clave que suelen incluirse en este tipo de textos:

1. **Vectores**: Se introducen los conceptos básicos de los vectores, sus propiedades y operaciones como la suma y la multiplicación por un escalar.

2. **Espacios Vectoriales**: Se define qué es un espacio vectorial y se discuten sus axiomas, así como ejemplos de espacios vectoriales comunes.

3. **Subespacios**: Se exploran los subespacios de un espacio vectorial, incluyendo criterios para determinar si un conjunto de vectores forma un subespacio.

4. **Combinaciones Lineales**: Se explica el concepto de combinaciones lineales y se introduce la noción de dependencia e independencia lineal de vectores.

5. **Bases y Dimensiones**: Se aborda el concepto de base de un espacio vectorial y cómo se relaciona con la dimensión del espac

In [ ]:
question10 = "Qué es un vector?"
answer10 = get_answer_with_cache(question10, index, documents, base_prompt, chat_history)
print(answer10)

Un vector es un objeto matemático que tiene tanto magnitud (o longitud) como dirección. En el contexto del álgebra lineal y la geometría, los vectores se utilizan para representar cantidades que tienen estas dos características. 

### Características de un vector:

1. **Magnitud**: Es la longitud del vector. Por ejemplo, en un espacio bidimensional, la magnitud de un vector \(\mathbf{v} = (x, y)\) se puede calcular usando la fórmula:
   \[
   \|\mathbf{v}\| = \sqrt{x^2 + y^2}
   \]

2. **Dirección**: Indica hacia dónde apunta el vector. Por ejemplo, un vector que apunta hacia el norte tiene una dirección diferente a uno que apunta hacia el este.

### Representación:

- **En el plano**: Un vector en \( \mathbb{R}^2 \) se puede representar como un par ordenado \((x, y)\), donde \(x\) es la componente horizontal y \(y\) es la componente vertical. Por ejemplo, el vector \((3, 4)\) tiene una magnitud y apunta en una dirección específica en el plano.

- **En el espacio tridimensional**: Un v

In [ ]:
question11 = "Da un ejemplo de un conjunto con un orden parcial que no sea total."
answer11 = get_answer_with_cache(question11, index, documents, base_prompt, chat_history)
print(answer11)

¡Claro! Para entender mejor lo que es un orden parcial y un orden total, primero definamos ambos conceptos.

Un **orden parcial** en un conjunto es una relación que cumple con tres propiedades: reflexividad, antisimetría y transitividad. Esto significa que, para cualquier par de elementos en el conjunto, no necesariamente se puede comparar uno con el otro.

Por otro lado, un **orden total** es un tipo especial de orden parcial donde, para cualquier par de elementos, uno de ellos es comparable con el otro.

Ahora, un ejemplo clásico de un conjunto con un orden parcial que no es total es el conjunto de subconjuntos de un conjunto dado, utilizando la inclusión como relación de orden.

Considera el conjunto \( S = \{ a, b \} \). Los subconjuntos de \( S \) son:

- \( \emptyset \) (el conjunto vacío)
- \( \{ a \} \)
- \( \{ b \} \)
- \( \{ a, b \} \)

La relación de inclusión \( \subseteq \) es un orden parcial. Por ejemplo:

- \( \emptyset \subseteq \{ a \} \)
- \( \emptyset \subseteq \{ b

In [ ]:
question12 = "¿Por qué no puede existir “el conjunto de todos los conjuntos”. ¿Qué tipo de contradicciones aparecen si se asume su existencia?"
answer12 = get_answer_with_cache(question12, index, documents, base_prompt, chat_history)
print(answer12)

¡Esa es una pregunta muy interesante y profunda! La idea del "conjunto de todos los conjuntos" se relaciona con conceptos fundamentales en teoría de conjuntos y lógica. Vamos a desglosarlo paso a paso.

1. **Definición de conjunto**: En teoría de conjuntos, un conjunto es una colección de objetos. Sin embargo, si existiera un "conjunto de todos los conjuntos", incluiría a sí mismo como un elemento.

2. **La paradoja de Russell**: Bertrand Russell formuló una famosa paradoja que ilustra el problema. Supongamos que existe un conjunto \( R \) que contiene todos los conjuntos que no se contienen a sí mismos. La pregunta es: ¿se contiene \( R \) a sí mismo? 

   - Si \( R \) se contiene a sí mismo, entonces, por definición, no debería estar en \( R \) (porque solo incluye conjuntos que no se contienen a sí mismos).
   - Si \( R \) no se contiene a sí mismo, entonces, según la definición, debería estar en \( R \) (porque es un conjunto que no se contiene a sí mismo).

   Esta contradicción m

In [ ]:
question13 = "Define una relación en Z que divida en clases según el residuo módulo 4. Escribe al menos dos clases."
answer13 = get_answer_with_cache(question13, index, documents, base_prompt, chat_history)
print(answer13)

¡Claro! Vamos a trabajar en esto paso a paso.

Primero, recordemos que cuando hablamos de "residuo módulo 4", nos referimos a los restos que obtenemos al dividir un número entero entre 4. Los posibles residuos son 0, 1, 2 y 3. Esto significa que podemos clasificar los números enteros en 4 clases diferentes según el residuo que dejan al ser divididos por 4.

### Definición de la relación

Podemos definir una relación \( R \) en \( \mathbb{Z} \) (el conjunto de los números enteros) de la siguiente manera:

Para \( a, b \in \mathbb{Z} \), decimos que \( a R b \) si \( a \equiv b \mod 4 \). Esto significa que \( a \) y \( b \) tienen el mismo residuo cuando se dividen por 4.

### Clases de equivalencia

Ahora, según esta relación, podemos definir las clases de equivalencia:

1. **Clase de equivalencia de 0**: Todos los números que dejan un residuo de 0 al dividirse por 4. Esta clase se puede representar como:
   \[
   [0] = \{ n \in \mathbb{Z} \mid n \equiv 0 \mod 4 \} = \{ \ldots, -8, -4,